In [2]:
# importação das bibliotecas necessárias para o as análises solicitadas"
import pandas as pd
import plotly.express as px
import streamlit as st

#carrge ar o dataset para análise
car_analysis = pd.read_csv("../vehicles.csv") 

#Imprimindo uma versão resumida do dataset para verificar as colunas disponíveis e os tipos de dados
print(car_analysis.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  object 
 3   condition     51525 non-null  object 
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  object 
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  object 
 8   type          51525 non-null  object 
 9   paint_color   42258 non-null  object 
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  object 
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), object(7)
memory usage: 5.1+ MB
None


##### **Conclusões iniciais**  
Após estudar o conteúdo do ficheiro csv, considero que para análise exploratória os dados mais importantes são os presentes - por ordem de importância  - nas colunas:  
1. Price  
2. Kilometragem
3. Ano
4. Estado
5. Tamanho/ tipo 
6. Depois marca/modelo, transmissão, combustível cor etc

Após analisar o preço, verifiquei que existem 3 442 valores de preço diferentes num dataset de 51 525 carros. Entre esses preços completamente distorcidos da realidade. Então vou fazer uma primeira limpeza considerando o seguinte   
1. Viaturas com preço < 2000
2. em bom estado (excellent, like_new, new)
3. com menos de 150 000 km
são valores incoerentes com o mercado real.

In [3]:
# Definição das condições
cond_preço = car_analysis['price'] < 2000
cond_estado = car_analysis['condition'].isin([ 'excellent', 'like_new', 'new'])
cond_km = car_analysis['odometer'] < 150000

# Combinação das condições para identificar os carros incoerentes
filter_incoerentes = cond_preço & cond_estado & cond_km

# Quantos carros incoerentes existem
car_analysis[filter_incoerentes].shape



(929, 13)

In [4]:
car_analysis[filter_incoerentes][['price', 'condition', 'odometer']].sample(15)


,price,condition,odometer
11661,1,excellent,12500.0
12189,1,excellent,78033.0
28935,1,excellent,42112.0
14329,1,excellent,47139.0
14630,1,excellent,8530.0
16327,1,excellent,20627.0
18852,1500,excellent,61000.0
49775,1999,excellent,77000.0
14007,1,excellent,8912.0
12104,1,excellent,89606.0


In [5]:
car_analysis = car_analysis[~filter_incoerentes]
car_analysis[filter_incoerentes].shape

C:\Users\Gerson\AppData\Local\Temp\ipykernel_24488\1423159929.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  car_analysis[filter_incoerentes].shape


(0, 13)

In [10]:
car_analysis['date_posted'] = pd.to_datetime(car_analysis['date_posted'], format='%Y-%m-%d') # Convertendo a coluna 'date_posted' de objeto para o formato datetime

car_analysis.info() # Exibindo informações sobre o DataFrame



<class 'pandas.core.frame.DataFrame'>
Index: 50596 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         50596 non-null  int64         
 1   model_year    47038 non-null  float64       
 2   model         50596 non-null  object        
 3   condition     50596 non-null  object        
 4   cylinders     45432 non-null  float64       
 5   fuel          50596 non-null  object        
 6   odometer      42704 non-null  float64       
 7   transmission  50596 non-null  object        
 8   type          50596 non-null  object        
 9   paint_color   41504 non-null  object        
 10  is_4wd        24961 non-null  float64       
 11  date_posted   50596 non-null  datetime64[ns]
 12  days_listed   50596 non-null  int64         
dtypes: datetime64[ns](1), float64(4), int64(2), object(6)
memory usage: 5.4+ MB


Após a remoção das viaturas com valores de venda "anormais" e a conversão da coluna com datas para o formato datetime, só falta fazer uma última "limpeza ao ficheiro:  
1. Na coluna is_4wd, transformar os dados em objet e substituir os valores 1.0 por 'yes' e os valores nulos em 'no' (não é relevante mas é um exercício interessante)
2. Ignorar os ausentes nas colunas paint_color e cylinders pois não são muito relevantes para as análises que pretendo fazer
3. Eliminar todas as linhas em que o model_year e odometer estão ausentes pois são dois dos factores determinantes na análise

In [ ]:
# preenchimentos dos dodos ausentes na coluna is_4wd
car_analysis['is_4wd'] = car_analysis['is_4wd'].fillna(0) # preenchendo os valores ausentes com 0, indicando que o carro não é 4wd (tração nas quatro rodas)
car_analysis['is_4wd'] = car_analysis['is_4wd'].replace({'yes': 1, 'no': 0}) # convertendo os valores 'yes' para 1 e 'no' para 0

# eliminação das linhas com valores ausentes na coluna 'model_year' e 'odometer' para garantir que todas as análises futuras tenham informações completas sobre o ano do modelo dos carros e a quilometragem
car_analysis = car_analysis.dropna(subset=['model_year','odometer']) # removendo as linhas onde a coluna 'model_year' e 'odometer' têm valores ausentes, garantindo que todas as análises futuras tenham informações completas sobre o ano do modelo dos carros
car_analysis.info() # Exibindo informações sobre o DataFrame após as limpezas e transformações realizadas

<class 'pandas.core.frame.DataFrame'>
Index: 39695 entries, 0 to 51523
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         39695 non-null  int64         
 1   model_year    39695 non-null  float64       
 2   model         39695 non-null  object        
 3   condition     39695 non-null  object        
 4   cylinders     35638 non-null  float64       
 5   fuel          39695 non-null  object        
 6   odometer      39695 non-null  float64       
 7   transmission  39695 non-null  object        
 8   type          39695 non-null  object        
 9   paint_color   32599 non-null  object        
 10  is_4wd        39695 non-null  int64         
 11  date_posted   39695 non-null  datetime64[ns]
 12  days_listed   39695 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(6)
memory usage: 4.2+ MB


C:\Users\Gerson\AppData\Local\Temp\ipykernel_24488\689893577.py:3: FutureWarning:

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



Agora sim estou satisfeito com os dados, pronto para executar as análises gráficas, com um novo ficheiro que vou criar: vehicules_updated.csv 

In [17]:
car_analysis = car_analysis.reset_index(drop=True) # Resetando o índice do DataFrame para garantir que ele seja sequencial e sem lacunas após a remoção dos carros incoerentes e das linhas com valores ausentes, facilitando futuras análises e manipulações dos dados.
car_analysis.to_csv("../vehicles_updated.csv", index=False) # Salvando o DataFrame limpo em um novo arquivo CSV para mantendo o original intacto e para facilitar futuras análises sem a necessidade de repetir o processo de limpeza. O arquivo é salvo com o nome "vehicles_updated.csv" e sem o índice para manter a estrutura do DataFrame.

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"
pio.templates.default = "plotly" # pedi estes comandos ao IA porque os gráficos estavam sem cores

condition_pie = px.pie(car_analysis,names="condition", title="car distribution by condition") # criação do gráfico de pizza usando plotly express

condition_pie.show() # apresentação do gráfico


In [46]:
fig = px.scatter(
    car_analysis,
    x="model_year",
    y="price",
    color="condition",
    title="Ano do Modelo vs Quilometragem (cor representa o preço)"
)

fig.show()